# Stage 4: KAN-Phase Implementation
**Environment:** Kaggle Notebook (GPU T4)  
**Requires:** `pip install pykan`

In [ ]:
!pip install pykan -q

In [ ]:
import numpy as np, pandas as pd, os, glob, json, time
from typing import Dict, List
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from kan import KAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix, roc_curve
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.notebook import tqdm

%matplotlib inline
plt.rcParams['figure.dpi'] = 120; sns.set_style('whitegrid')

INPUT_DIR = '/kaggle/input/artifact-dataset'
OUTPUT_DIR = '/kaggle/working'
CACHE_DIR = os.path.join(OUTPUT_DIR, 'phase_cache')
MODEL_DIR = os.path.join(OUTPUT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

CONFIG = {
    'n_pca_components': 128, 'kan_width': [128, 64, 32, 1],
    'kan_grid_size': 5, 'kan_spline_order': 3,
    'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64,
    'epochs': 50, 'early_stopping_patience': 10, 'random_state': 42,
}

In [ ]:
# Cell 2: Load & Prepare Data
cache_files = sorted(glob.glob(os.path.join(CACHE_DIR, 'phase_maps_*.npy')))
phase_results = np.load(cache_files[-1], allow_pickle=True).item()
meta_files = glob.glob(os.path.join(INPUT_DIR, '**', 'metadata.csv'), recursive=True)
metadata_df = pd.concat([pd.read_csv(mf).assign(generator=os.path.basename(os.path.dirname(mf)))
                         for mf in meta_files], ignore_index=True) if meta_files else None

X_list, y_list = [], []
for gen, maps in phase_results.items():
    if metadata_df is not None:
        gen_df = metadata_df[metadata_df['generator'] == gen]
        is_real = len(gen_df) > 0 and gen_df['target'].mode().iloc[0] == 0
    else: is_real = 'real' in gen.lower()
    for m in maps: X_list.append(m.flatten()); y_list.append(0 if is_real else 1)

X_raw = np.array(X_list, dtype=np.float32); y = np.array(y_list, dtype=np.int64)
scaler = StandardScaler(); pca = PCA(n_components=CONFIG['n_pca_components'], random_state=42)
X_pca = pca.fit_transform(scaler.fit_transform(X_raw)).astype(np.float32)
print(f'PCA: {X_pca.shape}, variance: {pca.explained_variance_ratio_.sum():.3f}')

X_tv, X_test, y_tv, y_test = train_test_split(X_pca, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.15, stratify=y_tv, random_state=42)
print(f'Train:{len(X_train)} Val:{len(X_val)} Test:{len(X_test)}')

train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train)), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val)), batch_size=64)
test_loader = DataLoader(TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test)), batch_size=64)

In [ ]:
# Cell 3: Train KAN-Phase
kan_model = KAN(width=CONFIG['kan_width'], grid=CONFIG['kan_grid_size'],
                k=CONFIG['kan_spline_order'], device=str(DEVICE))
n_params = sum(p.numel() for p in kan_model.parameters())
print(f'KAN: {CONFIG["kan_width"]}, grid={CONFIG["kan_grid_size"]}, params={n_params:,}')

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(kan_model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
history = {'train_loss':[], 'val_loss':[], 'train_auc':[], 'val_auc':[]}
best_auc, patience_ctr = 0.0, 0

for epoch in tqdm(range(CONFIG['epochs']), desc='KAN Training'):
    kan_model.train(); t_loss, t_p, t_t = 0, [], []
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.float().to(DEVICE)
        optimizer.zero_grad(); logits = kan_model(X_b).squeeze(-1)
        loss = criterion(logits, y_b); loss.backward(); optimizer.step()
        t_loss += loss.item()*len(y_b)
        t_p.append(torch.sigmoid(logits).detach().cpu().numpy()); t_t.append(y_b.cpu().numpy())
    t_p, t_t = np.concatenate(t_p), np.concatenate(t_t)
    t_auc = roc_auc_score(t_t, t_p) if len(np.unique(t_t))>1 else 0
    
    kan_model.eval(); v_loss, v_p, v_t = 0, [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.float().to(DEVICE)
            logits = kan_model(X_b).squeeze(-1); v_loss += criterion(logits, y_b).item()*len(y_b)
            v_p.append(torch.sigmoid(logits).cpu().numpy()); v_t.append(y_b.cpu().numpy())
    v_p, v_t = np.concatenate(v_p), np.concatenate(v_t)
    v_auc = roc_auc_score(v_t, v_p) if len(np.unique(v_t))>1 else 0
    scheduler.step(v_auc)
    history['train_loss'].append(t_loss/len(train_loader.dataset))
    history['val_loss'].append(v_loss/len(val_loader.dataset))
    history['train_auc'].append(t_auc); history['val_auc'].append(v_auc)
    if v_auc > best_auc:
        best_auc = v_auc; patience_ctr = 0
        torch.save(kan_model.state_dict(), os.path.join(MODEL_DIR, 'kan_phase_best.pth'))
    else:
        patience_ctr += 1
        if patience_ctr >= CONFIG['early_stopping_patience']:
            print(f'Early stop at epoch {epoch+1}'); break

kan_model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'kan_phase_best.pth'), weights_only=True))
print(f'Best val AUC: {best_auc:.4f}')

In [ ]:
# Cell 4: Evaluation & A2 Comparison
kan_model.eval(); all_p, all_t = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        all_p.append(torch.sigmoid(kan_model(X_b.to(DEVICE)).squeeze(-1)).cpu().numpy()); all_t.append(y_b.numpy())
kan_preds, kan_targets = np.concatenate(all_p), np.concatenate(all_t)
kan_acc = accuracy_score(kan_targets, (kan_preds>0.5).astype(int))
kan_auc = roc_auc_score(kan_targets, kan_preds)

print(f'KAN-Phase: Accuracy={kan_acc:.4f}, AUC={kan_auc:.4f}, Params={n_params:,}')
print(classification_report(kan_targets, (kan_preds>0.5).astype(int), target_names=['Real','Fake']))

# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ep = range(1, len(history['train_loss'])+1)
ax1.plot(ep, history['train_loss'], label='Train'); ax1.plot(ep, history['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(ep, history['train_auc'], label='Train'); ax2.plot(ep, history['val_auc'], label='Val')
ax2.set_title('AUC'); ax2.legend()
plt.suptitle('KAN-Phase Training', fontweight='bold'); plt.tight_layout(); plt.show()

# A2: Load MLP results for comparison
b3_path = os.path.join(MODEL_DIR, 'results_b3.json')
if os.path.exists(b3_path):
    with open(b3_path) as f: b3 = json.load(f)
    print(f'\n{"="*50}\nABLATION A2: KAN vs MLP\n{"="*50}')
    comp = pd.DataFrame([{'Model':'KAN-Phase','AUC':kan_auc,'Acc':kan_acc,'Params':n_params},
                         {'Model':b3['model'],'AUC':b3['test_auc'],'Acc':b3['test_accuracy'],'Params':b3['n_parameters']}])
    print(comp.to_string(index=False))
    comp.to_csv(os.path.join(MODEL_DIR, 'ablation_a2.csv'), index=False)

# ROC + Confusion
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fpr, tpr, _ = roc_curve(kan_targets, kan_preds)
ax1.plot(fpr, tpr, lw=2, label=f'KAN AUC={kan_auc:.3f}'); ax1.plot([0,1],[0,1],'k--',alpha=0.3)
ax1.set_title('ROC'); ax1.legend()
cm = confusion_matrix(kan_targets, (kan_preds>0.5).astype(int))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2, xticklabels=['Real','Fake'], yticklabels=['Real','Fake'])
ax2.set_title('KAN Confusion Matrix'); plt.tight_layout(); plt.show()

with open(os.path.join(MODEL_DIR, 'results_kan.json'), 'w') as f:
    json.dump({'model':'KAN-Phase','test_accuracy':float(kan_acc),'test_auc':float(kan_auc),
               'n_parameters':n_params,'kan_width':CONFIG['kan_width'],'kan_grid_size':CONFIG['kan_grid_size']}, f, indent=2)
print('KAN results saved.')